# Fine-tuning LLM sur TeleQnA (normes 3GPP)
## Google Colab (GPU T4 gratuit)

**1.** Runtime > Change runtime type > **T4 GPU**

**2.** Execute **Runtime > Run all**

**3.** Quand la cellule demande, uploade `telecom_train.json` puis `telecom_test_split.json`

In [ ]:
# ============================================================
# 1. Installer Unsloth (officiel)
# ============================================================
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft triton cut_cross_entropy unsloth_zoo
    !pip install trl==0.15.2 -q
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth
    !pip install sentence-transformers

import torch
print(f"GPU: {torch.cuda.get_device_name()}" if torch.cuda.is_available() else "ERREUR: pas de GPU!")

In [ ]:
# ============================================================
# 2. Uploader telecom_train.json
# ============================================================
from google.colab import files
import json

print("Etape 1/2: Telechargez telecom_train.json")
uploaded = files.upload()
train_path = [f for f in uploaded.keys() if f.endswith('.json')][0]
with open(train_path, encoding="utf-8") as f:
    train_data = json.load(f)
print(f"Charge: {len(train_data)} questions TeleQnA")

print("\nEtape 2/2: Telechargez telecom_test_split.json")
uploaded2 = files.upload()
test_path = [f for f in uploaded2.keys() if f.endswith('.json')][0]
with open(test_path, encoding="utf-8") as f:
    test_data = json.load(f)
print(f"Test: {len(test_data)} questions")

# Verifier format
assert "question" in test_data[0], f"Format test invalide: {list(test_data[0].keys())}"
assert "reponse" in test_data[0]
assert "options" in test_data[0]
print("Format OK")

In [ ]:
# ============================================================
# 3. Formater les donnees d'entrainement (QCM -> prompt)
# ============================================================
import re

def format_exemple(item):
    opts = "\n".join(f"{chr(65+i)}) {o}" for i, o in enumerate(item["options"]))
    m = re.search(r'option (\d+)', item["reponse"])
    reponse_letter = chr(65 + int(m.group(1)) - 1) if m else "A"
    prompt = f"""### Contexte:
Vous etes un expert en telecommunications 3GPP.
Repondez a la question QCM.

### Question:
{item['question']}

### Options:
{opts}

### Reponse: {reponse_letter}"""
    return {"text": prompt}

train_formatted = [format_exemple(item) for item in train_data]
print(f"Formate: {len(train_formatted)} exemples")

In [ ]:
# ============================================================
# 4. Charger le modele + Ajouter LoRA
# ============================================================
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/phi-3-mini-4k-instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

params = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Modele: {MODEL_NAME} ({params:.1f}B parametres)")

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f"Parametres entrainables: {trainable:.1f}M")

In [ ]:
# ============================================================
# 5. Split + Entrainement
# ============================================================
from unsloth import is_bfloat16_supported
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

dataset = Dataset.from_list(train_formatted)
split = dataset.train_test_split(test_size=0.2, seed=42)
train_val = split["train"]
eval_ds = split["test"]
print(f"Train: {len(train_val)}, Validation: {len(eval_ds)}")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_val,
    eval_dataset=eval_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

print("Entrainement... (~3h pour 3 epoques sur Phi-3 / T4)")
trainer.train()

In [ ]:
# ============================================================
# 6. Evaluation sur telecom_test_split.json
# ============================================================
import time

FastLanguageModel.for_inference(model)
model.generation_config.max_length = None

nb_test = min(50, len(test_data))
test_subset = test_data[:nb_test]
print(f"Evaluation Fine-tuning sur {nb_test} questions...")

ft_results = []

for idx, item in enumerate(test_subset):
    question = item["question"]
    attendue = item["reponse"]
    options = item.get("options", [])

    # Formater le prompt SANS la reponse
    opts = "\n".join(f"{chr(65+i)}) {o}" for i, o in enumerate(options))
    prompt = f"""### Contexte:
Vous etes un expert en telecommunications 3GPP.
Repondez a la question QCM.

### Question:
{question}

### Options:
{opts}

### Reponse:"""

    start = time.time()
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=4, temperature=0.1, do_sample=True)
    elapsed = time.time() - start

    reponse_brute = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    # Extraire la lettre (A/B/C/D) de la prediction
    m_letter = re.search(r'[A-D]', reponse_brute)
    pred_letter = m_letter.group(0) if m_letter else ""

    # Lettre correcte
    m_correct = re.match(r"option\s+(\d+)", attendue.lower())
    opt_num = m_correct.group(1) if m_correct else None
    correct_letter = chr(65 + int(opt_num) - 1) if opt_num else ""

    # Exact Match
    exact = correct_letter == pred_letter

    ft_results.append({
        "question": question,
        "reponse_attendue": attendue,
        "reponse_predite": reponse_brute,
        "modele": "fine-tuned (Phi-3)",
        "approche": "Fine-tuning",
        "exact_match": exact,
        "temps": round(elapsed, 3),
    })

    if (idx + 1) % 10 == 0:
        print(f"  {idx+1}/{nb_test} traitees")

n = len(ft_results)
em = sum(1 for r in ft_results if r["exact_match"])
temps_moy = sum(r["temps"] for r in ft_results) / n
print(f"\nResultats Fine-tuning sur {n} questions:")
print(f"  Exact Match: {em}/{n} ({em/n*100:.1f}%)")
print(f"  Temps moyen: {temps_moy:.3f}s")

In [ ]:
# ============================================================
# 7. Calculer le F1 semantique (meme metrique que le projet local)
# ============================================================
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

def _semantic_similarity(text1, text2):
    emb = embedder.encode([text1, text2])
    emb = emb / np.linalg.norm(emb, axis=1, keepdims=True)
    return float(np.dot(emb[0], emb[1]))

def _clean_prediction(text):
    text = text.strip()
    patterns = [
        r"^[Bb]ased\s+on\s+(the\s+)?(context|document|specification)[^:]*:?\s*",
        r"^[Tt]he\s+(answer|correct|right)\s+(is|option|choice)[^:]*:?\s*",
        r"^[Aa]ccording\s+to\s+(the\s+)?(context|document)[^:]*:?\s*",
        r"^[Oo]ption\s+\d+\s*:\s*",
        r"^[Rr]eponse\s*:\s*",
        r"^[Aa]nswer\s*:\s*",
    ]
    for p in patterns:
        text = re.sub(p, "", text).strip()
    return text

for r in ft_results:
    attendue = r["reponse_attendue"]

    # Texte de l'option correcte
    m = re.match(r"option\s+(\d+)", attendue.lower())
    opt_num = m.group(1) if m else None
    correct_text = ""
    if opt_num:
        for q in test_data:
            if q["question"] == r["question"]:
                opts = q.get("options", [])
                idx = int(opt_num) - 1
                if 0 <= idx < len(opts):
                    correct_text = opts[idx].strip()
                break

    cleaned = _clean_prediction(r["reponse_predite"]) or r["reponse_predite"]
    compare = correct_text or (attendue.split(":", 1)[-1].strip() if ":" in attendue else attendue)
    f1_val = _semantic_similarity(cleaned, compare) if compare else 0.0

    threshold = 0.4
    r["f1_score"] = round(f1_val, 4)
    r["precision"] = round(1.0 if f1_val >= threshold else 0.0, 4)
    r["recall"] = round(1.0 if f1_val >= threshold else 0.0, 4)

f1_moy = sum(r["f1_score"] for r in ft_results) / n
print(f"  F1 moyen: {f1_moy:.4f}")
print(f"  Precision: {sum(r['precision'] for r in ft_results) / n:.4f}")
print(f"  Recall: {sum(r['recall'] for r in ft_results) / n:.4f}")

In [ ]:
# ============================================================
# 8. Sauvegarder sur Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

save_path = "/content/drive/MyDrive/telecom_lora_adapter"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Adapter LoRA sauvegarde dans: {save_path}")

# Metriques par epoque
import glob
state_files = sorted(glob.glob("outputs/checkpoint-*/trainer_state.json"))
if state_files:
    with open(state_files[-1]) as f:
        state = json.load(f)
    epoch_metrics = []
    for log in state.get("log_history", []):
        if "eval_loss" in log:
            epoch_metrics.append({"epoch": log["epoch"], "eval_loss": round(log["eval_loss"], 4)})
    with open(f"{save_path}/metriques_par_epoch.json", "w") as f:
        json.dump(epoch_metrics, f, indent=2)
    print(f"Metriques par epoque ({len(epoch_metrics)}):")
    for m in epoch_metrics:
        print(f"  Epoch {m['epoch']:.0f}: loss={m['eval_loss']:.4f}")

# Resultats de comparaison (format compatible avec comparer_approches.py)
resume_ft = [{
    "modele_approche": "fine-tuned (Phi-3) / Fine-tuning",
    "modele": "fine-tuned (Phi-3)",
    "approche": "Fine-tuning",
    "exact_match_rate": round(em / n, 4),
    "f1_moyen": round(f1_moy, 4),
    "precision": round(sum(r['precision'] for r in ft_results) / n, 4),
    "recall": round(sum(r['recall'] for r in ft_results) / n, 4),
    "temps_moyen": round(temps_moy, 3),
    "nb_questions": n,
}]

output = {
    "resume": resume_ft,
    "details": {"fine-tuned (Phi-3) / Fine-tuning": ft_results},
}

with open(f"{save_path}/resultats_finetuning.json", "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)
print(f"\nResultats sauvegardes dans: {save_path}/resultats_finetuning.json")
print(f"  EM={em/n*100:.1f}% | F1={f1_moy:.4f} | Temps={temps_moy:.3f}s | n={n}")

In [ ]:
# ============================================================
# 9. (Optionnel) Exporter en GGUF
# ============================================================
# model.save_pretrained_gguf("telecom_gguf", tokenizer, quantization_method="q4_k_m")
# print("GGUF sauvegarde dans telecom_gguf/")
# from google.colab import files
# files.download("telecom_gguf/telecom_gguf-unsloth.Q4_K_M.gguf")

## Apres l'execution

1. L'adapter LoRA est dans **Google Drive** > `telecom_lora_adapter/`
2. Les resultats sont dans `resultats_finetuning.json`
3. Copie ce fichier dans `project-rag/reports/`
4. Execute `python scripts/merger_ft_rag.py` pour fusionner
5. Execute `python scripts/generer_graphiques.py` pour mettre a jour les graphiques